# 02 · Tables and layout — what survives, what flattens

**A naive text walk over this same page doesn't error on its one table — it silently skips it, with no error and no placeholder.**

**In → out:** the same converted `DoclingDocument` from `01-pdf-printed` →
a per-element block-type map (heading / paragraph / table / figure) and a
cropped PNG per table/figure.

This notebook builds a `DocItemLabel` → block-type map and walks the
document region-by-region, plus a cropped-asset export for each
table/figure. The region walk returns plain dicts rather than a richer
model class — there's no need for a dedicated `Region`/`Line` type when a
flat dict carries the same fields for this stage's purposes.

The output-directory bookkeeping used at the end of this notebook is kept
separate from layout *regions* (despite living in similarly-named code) —
it's just about where a run's artifacts land on disk. No API key is
needed — everything here runs offline against a document converted
locally by docling.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `build_pdf_converter` | Rebuilds the docling converter from `01-pdf-printed`, trimmed to what this notebook needs, so it runs standalone | `build_pdf_converter(repo_root=REPO_ROOT)` |
| `document_to_blocks` | Walks a converted `DoclingDocument` into block-type dicts (heading/paragraph/table/figure) via `_DOCLING_TYPE_MAP` | `document_to_blocks(document)` |
| `export_table_and_figure_assets` | Crops and saves a PNG for each table/figure region in a document | `export_table_and_figure_assets(document, assets_dir, "printed-page")` |
| `resolve_paper_output_dir` | Resolves an output directory for a paper's artifacts, flat or per-paper folder | `resolve_paper_output_dir(RUN_DIR, "printed-page", layout="folder")` |
| `paper_artifact_paths` | Returns the standard artifact file paths (docling json, assets dir, manifest json) for an output dir | `paper_artifact_paths(out_dir, "printed-page")` |

## Step 1 — locate the repo root and confirm the environment

Before anything else, resolve `REPO_ROOT` (the kernel's cwd is this notebook's own directory, not the repo root) and print which API keys are present, so the offline/no-key path this notebook takes is stated up front rather than discovered by a later failure.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")

import nbio

REPO_ROOT = nbio.bootstrap()
nbio.show_environment()

## Step 2 — rebuild the PDF converter

Same converter as `01-pdf-printed`, rebuilt here (trimmed to just what this
notebook needs) so this notebook runs standalone rather than depending on
another notebook's kernel state. OCR stays off — `sample-data/printed-page.pdf`
is born-digital.

In [ ]:
from typing import Any, Literal

Device = Literal["auto", "cpu", "cuda", "mps"]


def build_pdf_converter(*, repo_root: Path, device: Device = "auto"):
    """Trimmed to what this notebook needs — see `01-pdf-printed.ipynb` for
    the full OCR-engine/accelerator resolution this is cut from."""
    if not os.environ.get("HF_HOME") and not os.environ.get("HUGGINGFACE_HUB_CACHE"):
        cache_dir = repo_root / ".cache" / "huggingface"
        cache_dir.mkdir(parents=True, exist_ok=True)
        os.environ["HF_HOME"] = str(cache_dir)

    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import (
        AcceleratorDevice,
        AcceleratorOptions,
        PdfPipelineOptions,
        TableFormerMode,
        TableStructureOptions,
    )
    from docling.document_converter import DocumentConverter, PdfFormatOption

    device_enum = {
        "auto": AcceleratorDevice.AUTO, "cpu": AcceleratorDevice.CPU,
        "cuda": AcceleratorDevice.CUDA, "mps": AcceleratorDevice.MPS,
    }[device]

    pipeline_options = PdfPipelineOptions(
        do_ocr=False,
        do_table_structure=True,
        # Needed so TableItem.get_image()/PictureItem.get_image() below have
        # a rendered page to crop from — without these, get_image() is a
        # silent None rather than an error.
        generate_page_images=True,
        generate_picture_images=True,
    )
    pipeline_options.table_structure_options = TableStructureOptions(
        do_cell_matching=True, mode=TableFormerMode.ACCURATE
    )
    pipeline_options.accelerator_options = AcceleratorOptions(device=device_enum)

    return DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)},
    )

## Step 3 — convert the sample page and look at real counts

`sample-data/printed-page.pdf` is the same synthetic, born-digital anchor
page `01-pdf-printed` uses. Convert it and print how many texts, tables and
pictures docling found — real output before anything downstream depends on
this `document`.

In [ ]:
SAMPLE_PDF = Path("sample-data/printed-page.pdf").resolve()
converter = build_pdf_converter(repo_root=REPO_ROOT)
result = converter.convert(str(SAMPLE_PDF))
document = result.document
print(f"texts={len(document.texts)} tables={len(document.tables)} pictures={len(document.pictures)}")

## Step 4 — the block-type map

`_DOCLING_TYPE_MAP` and `document_to_blocks` define the vocabulary a
downstream chunker reads — there's no application-specific logic in a
lookup table, just the mapping from docling's own labels to a simpler set
of block types. `Region`/`Line` are represented as plain dicts rather than
a heavier model class; RTL/script-specific bookkeeping is left out
entirely since this sample is a synthetic English test page with none of
that content. A table's `text` is deliberately left empty — its content
lives in `table_markdown` instead, which is exactly the distinction the
next step depends on.

In [ ]:
_DOCLING_TYPE_MAP: dict[str, str] = {
    "title": "heading",
    "section_header": "heading",
    "text": "paragraph",
    "paragraph": "paragraph",
    "caption": "paragraph",
    "footnote": "footnote",
    "table": "table",
    "picture": "figure",
    "figure": "figure",
    "list_item": "paragraph",
    "formula": "paragraph",
    "page_header": "heading",
    "page_footer": "footnote",
    "code": "paragraph",
}
_DEFAULT_BLOCK_TYPE = "paragraph"


def document_to_blocks(doc) -> list[dict]:
    """One dict per element, in reading order: {block_type, text,
    table_markdown, bbox}. A table's `text` is deliberately empty — its
    content lives in `table_markdown` instead; see the next cell for why
    that distinction is the whole point of this notebook.
    """
    blocks: list[dict] = []
    for item, _level in doc.iterate_items():
        label = getattr(getattr(item, "label", None), "value", None) or str(getattr(item, "label", "text"))
        block_type = _DOCLING_TYPE_MAP.get(label.lower(), _DEFAULT_BLOCK_TYPE)

        prov = getattr(item, "prov", None)
        bbox = None
        if prov:
            b = prov[0].bbox
            bbox = [round(b.l, 1), round(b.t, 1), round(b.r - b.l, 1), round(b.t - b.b, 1)]

        if block_type == "table":
            table_md = item.export_to_markdown(doc)
            blocks.append({
                "block_type": "table", "text": getattr(item, "text", "") or "",
                "table_markdown": table_md, "bbox": bbox,
            })
        elif block_type == "figure":
            blocks.append({"block_type": "figure", "text": "", "table_markdown": None, "bbox": bbox})
        else:
            blocks.append({
                "block_type": block_type, "text": getattr(item, "text", "") or "",
                "table_markdown": None, "bbox": bbox,
            })
    return blocks

## Step 5 — run `document_to_blocks` and look at real output

Walk the converted document and print one row per block: its type, how
much text it carries, and whether it carries table markdown instead.

In [ ]:
blocks = document_to_blocks(document)
nbio.table(
    [(b["block_type"], len(b["text"]), str(bool(b["table_markdown"]))) for b in blocks],
    headers=("block_type", "text_len", "has_table_markdown"),
)

## What survives, what flattens

The table on the sample page has five rows and three columns. Two ways to
turn this document into plain text:

1. **Structured** — walk every block, and when `block_type == "table"`,
   emit `table_markdown` (docling's TableFormer output: real rows, real
   columns, `|` cell boundaries a chunker can key on).
2. **Naive** — walk `document.texts` only, the way a text-first pipeline
   often does when tables aren't specifically handled, and join whatever
   has a non-empty `.text`.

`TableItem.text` is empty by construction (its content lives in
`table_markdown`), and `document.texts` never contains `TableItem`s in the
first place — so the naive walk doesn't mangle the table, it **skips it
outright**, with no error and no placeholder. That's the failure named at
the top of this stage's PRD: "a table flattened here can never be
retrieved... and it is the one failure a reader cannot detect downstream,
because the flattened text looks like text."

## Step 6 — compare the structured walk against the naive walk

Build both versions from the same `blocks` and `document.texts`, print
both, and check directly whether the table survived each one.

In [ ]:
structured = "\n\n".join(
    b["table_markdown"] if b["block_type"] == "table" else b["text"]
    for b in blocks if b["text"] or b["table_markdown"]
)
naive = "\n\n".join(t.text for t in document.texts if t.text.strip())

print("--- structured walk (table included) ---")
print(structured)
print()
print("--- naive walk over document.texts only ---")
print(naive)
print()
print("table present in structured output:", "Extract" in structured and "|" in structured)
print("table present in naive output:     ", "Extract" in naive)

## Step 7 — cropped table/figure assets

`export_table_and_figure_assets` crops and saves a PNG for each
table/figure region — unchanged except that it no longer feeds a run
manifest dict, which this stage doesn't carry.

In [ ]:
from docling_core.types.doc import PictureItem, TableItem


def export_table_and_figure_assets(document, assets_dir: Path, stem: str) -> dict[str, int]:
    assets_dir.mkdir(parents=True, exist_ok=True)
    table_count = 0
    picture_count = 0
    for element, _level in document.iterate_items():
        if isinstance(element, TableItem):
            image = element.get_image(document)
            if image is None:
                continue
            table_count += 1
            image.save(assets_dir / f"{stem}-table-{table_count:03d}.png", "PNG")
        elif isinstance(element, PictureItem):
            image = element.get_image(document)
            if image is None:
                continue
            picture_count += 1
            image.save(assets_dir / f"{stem}-figure-{picture_count:03d}.png", "PNG")
    return {"tables_exported": table_count, "figures_exported": picture_count}

## Step 8 — where the output lands

`resolve_paper_output_dir` and `paper_artifact_paths` handle output-
*directory* layout — where a paper's artifacts go on disk — not the
document layout regions the rest of this notebook works with. Defined
together since they're both used, and only used, right below.

In [ ]:
from typing import Literal

OutputLayout = Literal["flat", "folder"]


def resolve_paper_output_dir(base_output_dir: Path, stem: str, layout: OutputLayout = "flat") -> Path:
    base = base_output_dir.expanduser().resolve()
    return base / stem if layout == "folder" else base


def paper_artifact_paths(output_dir: Path, stem: str) -> dict[str, Path]:
    return {
        "docling_json": output_dir / f"{stem}.docling.json",
        "assets_dir": output_dir / f"{stem}.assets",
        "manifest_json": output_dir / f"{stem}.manifest.json",
    }

## Step 9 — export this run's cropped assets and look at real output

Resolve this run's output directory, export the table/figure crops into it,
and print what actually got written to disk.

In [ ]:
RUN_DIR = REPO_ROOT / "runs" / "01-extract-demo" / "extract"
out_dir = resolve_paper_output_dir(RUN_DIR, "printed-page", layout="folder")
paths = paper_artifact_paths(out_dir, "printed-page")

asset_counts = export_table_and_figure_assets(document, paths["assets_dir"], "printed-page")
print(asset_counts)
print("wrote:", sorted(p.name for p in paths["assets_dir"].glob("*.png")))

## What this stage covers, what it doesn't

| Kept | Left out | Why |
|---|---|---|
| `_DOCLING_TYPE_MAP` block-type vocabulary | a richer `Region`/`Line` model class | plain dicts carry the same fields here |
| cropped table/figure PNG export | RTL/script bookkeeping | no RTL content in this stage's sample |
| output-dir helpers | `is_extraction_complete` / `clear_partial_outputs` checkpoint logic | resume-on-crash behaviour belongs to a real multi-document run, not a single demo page |

See `01-pdf-printed.ipynb` for the conversion this notebook reuses, and the
stage `README.md` for the PDF-only / image asymmetry this table-focused
notebook doesn't touch (it only ever sees the PDF path).